In [42]:
import numpy as np
import pandas as pd

In [43]:
df=pd.read_csv("eda_file")

In [44]:
df.head()

,target,text,num_char,words,num_sent
0,0,"Go until jurong point, crazy.. Available only ...",111,24,2
1,0,Ok lar... Joking wif u oni...,29,8,2
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,155,37,2
3,0,U dun say so early hor... U c already then say...,49,13,1
4,0,"Nah I don't think he goes to usf, he lives aro...",61,15,1


In [45]:
 # library import kr rha hu
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [46]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5169 entries, 0 to 5168
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   target    5169 non-null   int64 
 1   text      5169 non-null   object
 2   num_char  5169 non-null   int64 
 3   words     5169 non-null   int64 
 4   num_sent  5169 non-null   int64 
dtypes: int64(4), object(1)
memory usage: 202.0+ KB


target      0
text        0
num_char    0
words       0
num_sent    0
dtype: int64

In [41]:
X = df['text']   # input (text)
y = df['target'] # output (0/1)

In [47]:
# train test ko split kr rha hu
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [48]:
tfidf = TfidfVectorizer(stop_words='english')

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [49]:
# yaha model chek kr gy best konsa hai
# Naive Bayes
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB

gnd = GaussianNB()
mnb = MultinomialNB()
bnb = BernoulliNB()

# Linear Model
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000)

# SVM
from sklearn.svm import SVC
svc = SVC()

# Tree
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()

# Ensemble
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier

rf = RandomForestClassifier()
et = ExtraTreesClassifier()
adb = AdaBoostClassifier()
gb = GradientBoostingClassifier()

# Neighbors
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier()

In [50]:
# sab model ko name de rhe hai
models = {
    "GaussianNB": gnd,
    "MultinomialNB": mnb,
    "BernoulliNB": bnb,
    "LogisticRegression": lr,
    "SVM": svc,
    "DecisionTree": dt,
    "RandomForest": rf,
    "ExtraTrees": et,
    "AdaBoost": adb,
    "GradientBoost": gb,
    "KNN": knn
}

In [52]:
X_train_tfidf
X_test_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6924 stored elements and shape (1034, 7398)>

In [54]:
from sklearn.metrics import accuracy_score, precision_score

results = []

for name, model in models.items():
    
    # special case (GaussianNB + KNN needs dense array)
    if name in ["GaussianNB", "KNN"]:
        model.fit(X_train_tfidf.toarray(), y_train)
        y_pred = model.predict(X_test_tfidf.toarray())
    else:
        model.fit(X_train_tfidf, y_train)
        y_pred = model.predict(X_test_tfidf)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    
    results.append((name, acc, prec))
    
    print(f"{name}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print("-"*30)

GaussianNB
Accuracy: 0.8752
Precision: 0.5351
------------------------------
MultinomialNB
Accuracy: 0.9662
Precision: 1.0000
------------------------------
BernoulliNB
Accuracy: 0.9710
Precision: 1.0000
------------------------------
LogisticRegression
Accuracy: 0.9632
Precision: 0.9820
------------------------------
SVM
Accuracy: 0.9787
Precision: 0.9920
------------------------------
DecisionTree
Accuracy: 0.9652
Precision: 0.8978
------------------------------
RandomForest
Accuracy: 0.9720
Precision: 0.9915
------------------------------
ExtraTrees
Accuracy: 0.9739
Precision: 0.9836
------------------------------
AdaBoost
Accuracy: 0.8946
Precision: 0.9500
------------------------------
GradientBoost
Accuracy: 0.9623
Precision: 0.9818
------------------------------
KNN
Accuracy: 0.8985
Precision: 1.0000
------------------------------


In [55]:
# yaha hamne dekha ki bernoli accha hai per 100 me as 28 ko detect kr rhahai
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94       889
           1       1.00      0.28      0.43       145

    accuracy                           0.90      1034
   macro avg       0.95      0.64      0.69      1034
weighted avg       0.91      0.90      0.87      1034



In [56]:
# yaha ham model chek kr rhe hai ki sabse acha kon hai
from sklearn.metrics import classification_report

for name, model in models.items():
    
    if name in ["GaussianNB", "KNN"]:
        model.fit(X_train_tfidf.toarray(), y_train)
        y_pred = model.predict(X_test_tfidf.toarray())
    else:
        model.fit(X_train_tfidf, y_train)
        y_pred = model.predict(X_test_tfidf)
    
    print("Model:", name)
    print(classification_report(y_test, y_pred))
    print("="*50)

Model: GaussianNB
              precision    recall  f1-score   support

           0       0.97      0.88      0.92       889
           1       0.54      0.84      0.65       145

    accuracy                           0.88      1034
   macro avg       0.75      0.86      0.79      1034
weighted avg       0.91      0.88      0.89      1034

Model: MultinomialNB
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       889
           1       1.00      0.76      0.86       145

    accuracy                           0.97      1034
   macro avg       0.98      0.88      0.92      1034
weighted avg       0.97      0.97      0.96      1034

Model: BernoulliNB
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       889
           1       1.00      0.79      0.88       145

    accuracy                           0.97      1034
   macro avg       0.98      0.90      0.93      1034
weighted avg    

In [57]:
# model select kr rhe hai ab ham
from sklearn.svm import SVC

final_model = SVC()

final_model.fit(X_train_tfidf, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [58]:
import pickle

pickle.dump(tfidf, open("vectorizer.pkl", "wb"))
pickle.dump(final_model, open("model.pkl", "wb"))